In [ ]:

#@title 1. Configuración del Entorno
import os

# Clonar el repositorio si no existe
if not os.path.exists("Signal-Analysis-Vol-I"):
    !git clone https://github.com/ecundir/Signal-Analysis-Vol-I.git

# Cambiar a la carpeta del capítulo
%cd Signal-Analysis-Vol-I/capitulo_5
print("✅ Entorno configurado correctamente.")

En este experimento:

**Generaremos el Caos:** Crearemos una señal "sucia" donde tres frecuencias específicas están sepultadas bajo un ruido blanco masivo.

**Aplicaremos el Escáner:** Usaremos la Transformada Rápida de Fourier (FFT) para identificar exactamente en qué frecuencias está la interferencia.

**Ejecutaremos la Limpieza:** Diseñaremos un filtro de precisión para rescatar la información útil y eliminar el ruido, demostrando la utilidad práctica de la teoría de Fourier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# --- PARÁMETROS CONFIGURABLES ---
fs = 10000        # Frecuencia de muestreo (10 kHz)
duracion = 0.5    # Segundos
t = np.arange(0, duracion, 1/fs)

# Definimos 3 frecuencias "secretas" (en Hz)
f1, f2, f3 = 120, 1100, 3500

# Creamos la señal pura
señal_pura = 1.0 * np.sin(2 * np.pi * f1 * t) + \
             1.5 * np.sin(2 * np.pi * f2 * t) + \
             1.2 * np.sin(2 * np.pi * f3 * t)

# Añadimos ruido blanco de alta potencia (Desafío para el ojo humano)
amplitud_ruido = 6.0
ruido = amplitud_ruido * np.random.randn(len(t))
señal_sucia = señal_pura + ruido

# Visualización inicial (Dominio del Tiempo)
plt.figure(figsize=(14, 4))
plt.plot(t, señal_sucia, color='gray', alpha=0.6)
plt.title("SEÑAL EN EL TIEMPO: ¿Puedes ver los tonos ocultos?")
plt.xlabel("Tiempo [s]")
plt.ylabel("Amplitud")
plt.xlim(0, 0.05) # Zoom para ver el caos
plt.grid(True)
plt.show()

In [ ]:
# --- CÁLCULO DE LA TRANSFORMADA ---
N = len(señal_sucia)
X_fft = np.fft.fft(señal_sucia)
frecuencias = np.fft.fftfreq(N, 1/fs)

# Tomamos solo la mitad positiva del espectro
f_pos = frecuencias[:N//2]
X_mag = (2.0/N) * np.abs(X_fft[:N//2])

# Graficar el espectro
plt.figure(figsize=(14, 5))
plt.plot(f_pos, X_mag, color='blue')
plt.title("DOMINIO DE LA FRECUENCIA: Los tonos han sido revelados")
plt.xlabel("Frecuencia [Hz]")
plt.ylabel("Magnitud")
plt.grid(True)
plt.xticks(np.arange(0, 5001, 500)) # Marcas cada 500Hz
plt.show()

print(f"Detectando picos dominantes...")

In [ ]:
# --- DISEÑO DEL FILTRO PASA-BANDA ---
# Queremos dejar pasar solo lo que esté cerca de 1100 Hz
f_centro = 1100
ancho_banda = 200 # Margen de maniobra

# Filtro Butterworth
sos = signal.butter(10, [f_centro - ancho_banda/2, f_centro + ancho_banda/2],
                    btype='bandpass', fs=fs, output='sos')
señal_filtrada = signal.sosfilt(sos, señal_sucia)

# Comparación Final
plt.figure(figsize=(14, 6))
plt.subplot(2, 1, 1)
plt.plot(t[:500], señal_sucia[:500], label="Original con Ruido", color='red', alpha=0.5)
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(t[:500], señal_filtrada[:500], label="Señal Recuperada (1.1 kHz)", color='green')
plt.title("Resultado del Filtrado Selectivo")
plt.xlabel("Tiempo [s]")
plt.legend()
plt.tight_layout()
plt.show()